In [ ]:
"""
============================================================
  TELECOM CUSTOMER SUPPORT AGENT — Agent Foundations Demo
  Topics Covered:
    • Agent Loop        → LangGraph StateGraph cycle
    • Reasoning Cycle   → LLM think → act → observe
    • Planning Basics   → Intent classification & routing
    • Memory            → ConversationBufferMemory + short-term state
    • Tool Usage        → Custom tools for each telecom domain
============================================================
"""

import os
import json
import random
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, List, Optional, Literal
import operator
# pyrefly: ignore [missing-import]
from dotenv import load_dotenv

# ─────────────────────────────────────────────
# LangChain / LangGraph imports
# ─────────────────────────────────────────────
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [ ]:
# ─────────────────────────────────────────────
# SYNTHETIC DATASET
# ─────────────────────────────────────────────

CUSTOMERS = {
    "C001": {
        "name": "Ravi Kumar",
        "phone": "9876543210",
        "plan": "Unlimited Pro 599",
        "balance": 45.50,
        "sim_status": "active",
        "roaming_enabled": False,
        "last_recharge": "2026-04-28",
        "data_used_gb": 18.2,
        "data_limit_gb": 100.0,
        "bills": [
            {"month": "April 2026", "amount": 599.00, "status": "paid"},
            {"month": "March 2026", "amount": 599.00, "status": "paid"},
        ],
        "open_tickets": [],
    },
    "C002": {
        "name": "Priya Sharma",
        "phone": "9123456780",
        "plan": "Basic 199",
        "balance": 0.00,
        "sim_status": "inactive",
        "roaming_enabled": False,
        "last_recharge": "2026-03-15",
        "data_used_gb": 2.5,
        "data_limit_gb": 10.0,
        "bills": [
            {"month": "April 2026", "amount": 199.00, "status": "unpaid"},
        ],
        "open_tickets": ["TKT-2201"],
    },
    "C003": {
        "name": "Arjun Mehta",
        "phone": "9988776655",
        "plan": "International 999",
        "balance": 250.00,
        "sim_status": "active",
        "roaming_enabled": True,
        "last_recharge": "2026-05-01",
        "data_used_gb": 55.0,
        "data_limit_gb": 200.0,
        "bills": [
            {"month": "April 2026", "amount": 999.00, "status": "paid"},
        ],
        "open_tickets": [],
    },
}

RECHARGE_PLANS = [
    {"plan_id": "P01", "name": "Basic 199",        "price": 199,  "validity": 28,  "data_gb": 10,  "calls": "100 mins"},
    {"plan_id": "P02", "name": "Value 299",         "price": 299,  "validity": 28,  "data_gb": 30,  "calls": "300 mins"},
    {"plan_id": "P03", "name": "Unlimited Pro 599", "price": 599,  "validity": 56,  "data_gb": 100, "calls": "Unlimited"},
    {"plan_id": "P04", "name": "International 999", "price": 999,  "validity": 28,  "data_gb": 200, "calls": "Unlimited + Roaming"},
    {"plan_id": "P05", "name": "Data Booster 99",   "price": 99,   "validity": 7,   "data_gb": 5,   "calls": "No calls"},
]

TROUBLE_SOLUTIONS = {
    "no_signal":     ["Toggle Airplane Mode ON then OFF", "Restart device", "Check for network outage in your area", "Re-insert SIM card"],
    "slow_internet": ["Clear browser/app cache", "Check data usage limit", "Switch between 4G/5G manually", "Reset APN settings"],
    "call_drops":    ["Move to open area", "Check VoLTE settings", "Update device software", "Contact support if persists"],
    "sms_failed":    ["Check message centre number", "Restart device", "Clear SMS app cache"],
    "sim_not_found": ["Re-insert SIM firmly", "Clean SIM contacts", "Try SIM in another slot", "Visit store for SIM replacement"],
}

TICKET_DB = {
    "TKT-2201": {"issue": "Billing overcharge April 2026", "status": "open", "created": "2026-05-02", "customer_id": "C002"},
}


In [ ]:
# ─────────────────────────────────────────────
# TOOLS  (Tool Usage Concept)
# ─────────────────────────────────────────────

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up customer account details by customer ID."""
    if customer_id in CUSTOMERS:
        c = CUSTOMERS[customer_id]
        return json.dumps({
            "found": True,
            "name": c["name"],
            "phone": c["phone"],
            "plan": c["plan"],
            "balance": c["balance"],
            "sim_status": c["sim_status"],
            "roaming_enabled": c["roaming_enabled"],
            "last_recharge": c["last_recharge"],
            "data_used_gb": c["data_used_gb"],
            "data_limit_gb": c["data_limit_gb"],
            "open_tickets": c["open_tickets"],
        })
    return json.dumps({"found": False, "error": "Customer not found"})


@tool
def activate_sim(customer_id: str) -> str:
    """Activate the SIM card for a customer."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"success": False, "error": "Customer not found"})
    c = CUSTOMERS[customer_id]
    if c["sim_status"] == "active":
        return json.dumps({"success": False, "message": "SIM is already active"})
    if c["balance"] <= 0:
        return json.dumps({"success": False, "message": "Insufficient balance. Please recharge first."})
    CUSTOMERS[customer_id]["sim_status"] = "active"
    ref = f"SIM-ACT-{random.randint(100000, 999999)}"
    return json.dumps({"success": True, "message": "SIM activated successfully!", "reference": ref})


@tool
def get_recharge_plans() -> str:
    """Retrieve all available recharge plans."""
    return json.dumps({"plans": RECHARGE_PLANS})


@tool
def process_recharge(customer_id: str, plan_id: str) -> str:
    """Process a recharge for the customer with the selected plan."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"success": False, "error": "Customer not found"})
    plan = next((p for p in RECHARGE_PLANS if p["plan_id"] == plan_id), None)
    if not plan:
        return json.dumps({"success": False, "error": "Invalid plan ID"})
    CUSTOMERS[customer_id]["plan"] = plan["name"]
    CUSTOMERS[customer_id]["balance"] += plan["price"]
    CUSTOMERS[customer_id]["last_recharge"] = datetime.now().strftime("%Y-%m-%d")
    ref = f"RCH-{random.randint(100000, 999999)}"
    expiry = (datetime.now() + timedelta(days=plan["validity"])).strftime("%Y-%m-%d")
    return json.dumps({
        "success": True,
        "plan": plan["name"],
        "amount": plan["price"],
        "validity_until": expiry,
        "reference": ref,
    })


@tool
def enable_roaming(customer_id: str, enable: bool) -> str:
    """Enable or disable international roaming for a customer."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"success": False, "error": "Customer not found"})
    c = CUSTOMERS[customer_id]
    if "International" not in c["plan"] and enable:
        return json.dumps({"success": False, "message": "Current plan does not support roaming. Please upgrade to International 999."})
    CUSTOMERS[customer_id]["roaming_enabled"] = enable
    action = "enabled" if enable else "disabled"
    return json.dumps({"success": True, "message": f"Roaming {action} successfully for {c['name']}"})


@tool
def get_billing_info(customer_id: str) -> str:
    """Retrieve billing history and outstanding dues for a customer."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"found": False, "error": "Customer not found"})
    c = CUSTOMERS[customer_id]
    unpaid = [b for b in c["bills"] if b["status"] == "unpaid"]
    total_due = sum(b["amount"] for b in unpaid)
    return json.dumps({
        "name": c["name"],
        "bills": c["bills"],
        "total_due": total_due,
        "open_tickets": c["open_tickets"],
    })


@tool
def raise_billing_dispute(customer_id: str, month: str, reason: str) -> str:
    """Raise a billing dispute ticket for a customer."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"success": False, "error": "Customer not found"})
    ticket_id = f"TKT-{random.randint(1000, 9999)}"
    TICKET_DB[ticket_id] = {
        "issue": f"Billing dispute for {month}: {reason}",
        "status": "open",
        "created": datetime.now().strftime("%Y-%m-%d"),
        "customer_id": customer_id,
    }
    CUSTOMERS[customer_id]["open_tickets"].append(ticket_id)
    return json.dumps({
        "success": True,
        "ticket_id": ticket_id,
        "message": f"Dispute raised. Ticket {ticket_id} created. Our team will review within 48 hrs.",
    })


@tool
def troubleshoot_issue(issue_type: str) -> str:
    """
    Get troubleshooting steps for a technical issue.
    issue_type must be one of: no_signal, slow_internet, call_drops, sms_failed, sim_not_found
    """
    steps = TROUBLE_SOLUTIONS.get(issue_type)
    if not steps:
        available = list(TROUBLE_SOLUTIONS.keys())
        return json.dumps({"found": False, "available_issues": available})
    return json.dumps({"issue": issue_type, "steps": steps, "escalate": "If issue persists, call 198 or visit nearest store."})


@tool
def check_ticket_status(ticket_id: str) -> str:
    """Check the status of an existing support ticket."""
    ticket = TICKET_DB.get(ticket_id)
    if not ticket:
        return json.dumps({"found": False, "error": "Ticket not found"})
    return json.dumps({"found": True, "ticket_id": ticket_id, **ticket})



In [ ]:
# ─────────────────────────────────────────────
# AGENT STATE  (Memory Concept)
# ─────────────────────────────────────────────

class AgentState(TypedDict):
    """
    Short-term memory: everything the agent knows in this session.
    messages   → full conversation history (agent loop memory)
    customer_id → resolved customer, persisted across turns
    intent     → classified intent for planning/routing
    turn_count → tracks reasoning cycles
    """
    messages: Annotated[List[BaseMessage], operator.add]
    customer_id: Optional[str]
    intent: Optional[str]
    turn_count: int

In [ ]:
# ─────────────────────────────────────────────
# SYSTEM PROMPT  (Reasoning Cycle anchor)
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """You are NOVA, an intelligent Telecom Customer Support Agent for ConnectX Telecom.

## Your Capabilities
You help customers with:
1. **SIM Activation** — activate inactive SIMs, guide through the process
2. **Recharge Issues** — show plans, process recharges, check balance
3. **Roaming** — enable/disable international roaming, explain plan eligibility
4. **Billing Disputes** — fetch bills, raise dispute tickets, check ticket status
5. **Troubleshooting** — diagnose technical issues (signal, internet, calls, SMS)

## Reasoning Approach (Think → Act → Observe)
- THINK: Understand what the customer needs before acting
- ACT: Use the appropriate tool(s) to get real data
- OBSERVE: Interpret tool results and respond helpfully

## Memory Instructions
- Always ask for the Customer ID (format: C001, C002, C003) if not provided
- Remember the customer ID throughout the conversation
- Reference previous messages to maintain context

## Response Style
- Be empathetic, professional, and concise
- Always confirm actions before executing them (e.g., recharges)
- Provide reference numbers after successful operations
- If a tool fails, explain clearly and suggest alternatives

## Available Customer IDs (Demo): C001, C002, C003
"""

# ─────────────────────────────────────────────
# LLM + TOOLS SETUP
# ─────────────────────────────────────────────

tools = [
    lookup_customer,
    activate_sim,
    get_recharge_plans,
    process_recharge,
    enable_roaming,
    get_billing_info,
    raise_billing_dispute,
    troubleshoot_issue,
    check_ticket_status,
]

def get_llm():
    """Initialize LLM — uses OPENAI_API_KEY from environment."""
    load_dotenv("key.env")
    api_key = os.getenv("OPENAI_API_KEY", "")
    if not api_key:
        raise ValueError(
            "\n❌ OPENAI_API_KEY not set!\n"
            "   Export it: export OPENAI_API_KEY='sk-...'\n"
        )
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=api_key)
    return llm.bind_tools(tools)


# ─────────────────────────────────────────────
# GRAPH NODES  (Agent Loop + Planning)
# ─────────────────────────────────────────────

def agent_node(state: AgentState) -> AgentState:
    """
    AGENT NODE — Reasoning Cycle Core
    Receives state (messages + memory), calls LLM, returns next action.
    This is the 'Think' + 'Act' phase of the ReAct loop.
    """
    llm_with_tools = get_llm()

    # Build message list: system + history
    messages_to_send = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    # LLM reasons and decides: respond directly OR call a tool
    response = llm_with_tools.invoke(messages_to_send)

    # Planning: extract intent from first human message
    intent = state.get("intent")
    if not intent and state["messages"]:
        first_human = next((m.content for m in state["messages"] if isinstance(m, HumanMessage)), "")
        intent = _classify_intent(first_human)

    return {
        "messages": [response],
        "customer_id": state.get("customer_id"),
        "intent": intent,
        "turn_count": state.get("turn_count", 0) + 1,
    }


def _classify_intent(text: str) -> str:
    """
    Planning Basics — simple intent classifier.
    In production this would be an LLM call or fine-tuned classifier.
    """
    text = text.lower()
    if any(w in text for w in ["activate", "sim", "inactive"]):
        return "sim_activation"
    elif any(w in text for w in ["recharge", "plan", "balance", "top up"]):
        return "recharge"
    elif any(w in text for w in ["roam", "international", "abroad", "travel"]):
        return "roaming"
    elif any(w in text for w in ["bill", "charge", "dispute", "overcharge", "invoice"]):
        return "billing"
    elif any(w in text for w in ["signal", "internet", "slow", "drop", "sms", "network"]):
        return "troubleshooting"
    return "general"


def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """
    ROUTING LOGIC — Agent Loop Decision Point
    If the last AI message has tool_calls → go to tools node
    Otherwise → end (return response to user)
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"

In [ ]:
# ─────────────────────────────────────────────
# BUILD LANGGRAPH  (Agent Loop Graph)
# ─────────────────────────────────────────────

def build_agent() -> StateGraph:
    """
    Constructs the LangGraph StateGraph.

    Graph structure:
        [START] → agent_node ──tool_calls?──→ tool_node
                      ↑                           │
                      └───────────────────────────┘
                      (loop until no tool calls)
        agent_node → [END]  (when LLM gives final answer)
    """
    graph = StateGraph(AgentState)

    # Nodes
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))

    # Entry point
    graph.set_entry_point("agent")

    # Conditional edge: agent → tools OR end
    graph.add_conditional_edges(
        "agent",
        should_continue,
        {"tools": "tools", "end": END},
    )

    # After tools → always go back to agent (the loop)
    graph.add_edge("tools", "agent")

    return graph.compile()

In [ ]:
# ─────────────────────────────────────────────
# CONVERSATION MEMORY WRAPPER
# ─────────────────────────────────────────────

class TelecomSupportSession:
    """
    Session wrapper that maintains conversation memory across turns.
    Demonstrates: Persistent short-term memory within a session.
    """

    def __init__(self):
        self.app = build_agent()
        self.history: List[BaseMessage] = []
        self.customer_id: Optional[str] = None
        self.intent: Optional[str] = None
        self.turn_count: int = 0
        self.session_id = f"SESSION-{random.randint(10000, 99999)}"

    def chat(self, user_input: str) -> str:
        """Send a message and get a response, maintaining full memory."""
        self.history.append(HumanMessage(content=user_input))

        # Build state with full conversation memory
        state: AgentState = {
            "messages": self.history,
            "customer_id": self.customer_id,
            "intent": self.intent,
            "turn_count": self.turn_count,
        }

        # Run the agent graph
        result = self.app.invoke(state)

        # Update session memory
        new_messages = result["messages"][len(self.history):]
        self.history = result["messages"]
        self.customer_id = result.get("customer_id")
        self.intent = result.get("intent")
        self.turn_count = result.get("turn_count", self.turn_count)

        # Extract final AI response
        ai_response = ""
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                ai_response = msg.content
                break

        return ai_response

    def get_session_summary(self) -> dict:
        """Return session memory summary — demonstrates memory recall."""
        return {
            "session_id": self.session_id,
            "turns": self.turn_count,
            "customer_id": self.customer_id,
            "detected_intent": self.intent,
            "messages_in_memory": len(self.history),
        }


In [ ]:
# ─────────────────────────────────────────────
# DEMO SCENARIOS  (Auto-run showcase)
# ─────────────────────────────────────────────

DEMO_SCENARIOS = {
    "1": {
        "title": "SIM Activation",
        "description": "Customer C002 (Priya) has inactive SIM, needs recharge + activation",
        "turns": [
            "Hi, my SIM is not working. My customer ID is C002.",
            "What recharge plans are available?",
            "Please recharge with the Basic 199 plan.",
            "Now please activate my SIM.",
        ],
    },
    "2": {
        "title": "Roaming Setup",
        "description": "Customer C001 (Ravi) wants to enable roaming before travel",
        "turns": [
            "Hello, I'm travelling abroad next week. Customer ID C001.",
            "Can you enable international roaming for me?",
            "What plan do I need for roaming?",
        ],
    },
    "3": {
        "title": "Billing Dispute",
        "description": "Customer C002 (Priya) disputes an overcharge",
        "turns": [
            "I have been overcharged on my April bill. My ID is C002.",
            "Please show me my billing details.",
            "I want to raise a dispute for April 2026 overcharge.",
        ],
    },
    "4": {
        "title": "Network Troubleshooting",
        "description": "Customer C003 faces slow internet",
        "turns": [
            "My internet is very slow since morning. Customer ID C003.",
            "Can you help me troubleshoot slow internet?",
            "Also check my data usage please.",
        ],
    },
}


In [ ]:
# ─────────────────────────────────────────────
# MAIN INTERACTIVE LOOP
# ─────────────────────────────────────────────

def print_banner():
    banner = """
╔══════════════════════════════════════════════════════════════╗
║         NOVA — ConnectX Telecom Customer Support            ║
║            Powered by LangGraph + LangChain                 ║
╠══════════════════════════════════════════════════════════════╣
║  Agent Foundations Topics Demonstrated:                     ║
║  ✓ Agent Loop        → LangGraph StateGraph cycle           ║
║  ✓ Reasoning Cycle   → Think → Act (Tool) → Observe         ║
║  ✓ Planning Basics   → Intent classification & routing      ║
║  ✓ Memory            → Session memory across turns          ║
║  ✓ Tool Usage        → 9 domain-specific telecom tools      ║
╠══════════════════════════════════════════════════════════════╣
║  Demo Customer IDs: C001 (Ravi) | C002 (Priya) | C003      ║
╚══════════════════════════════════════════════════════════════╝
"""
    print(banner)


def run_demo_scenario(scenario_key: str):
    """Auto-run a demo scenario showing agent capabilities."""
    scenario = DEMO_SCENARIOS[scenario_key]
    print(f"\n{'═'*60}")
    print(f"  DEMO: {scenario['title']}")
    print(f"  {scenario['description']}")
    print(f"{'═'*60}\n")

    session = TelecomSupportSession()
    for turn in scenario["turns"]:
        print(f"👤 Customer: {turn}")
        print("🤖 NOVA: ", end="", flush=True)
        response = session.chat(turn)
        print(response)
        print()

    print(f"\n📊 Session Summary: {json.dumps(session.get_session_summary(), indent=2)}")


def interactive_chat():
    """Run interactive chat loop."""
    session = TelecomSupportSession()
    print("\n💬 Starting interactive session. Type 'quit' to exit, 'summary' for session info.\n")
    print("🤖 NOVA: Hello! I'm NOVA, your ConnectX Telecom support agent. How can I help you today?\n")

    while True:
        try:
            user_input = input("👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nSession ended.")
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "bye"):
            print("🤖 NOVA: Thank you for contacting ConnectX. Have a great day! 👋")
            break
        if user_input.lower() == "summary":
            print(f"\n📊 {json.dumps(session.get_session_summary(), indent=2)}\n")
            continue

        response = session.chat(user_input)
        print(f"🤖 NOVA: {response}\n")

In [ ]:
def main():
    print_banner()

    print("Select mode:")
    print("  [1] Demo: SIM Activation")
    print("  [2] Demo: Roaming Setup")
    print("  [3] Demo: Billing Dispute")
    print("  [4] Demo: Network Troubleshooting")
    print("  [5] Interactive Chat")
    print("  [6] Run ALL demos")

    try:
        choice = input("\nEnter choice (1-6): ").strip()
    except (EOFError, KeyboardInterrupt):
        choice = "6"

    if choice in ("1", "2", "3", "4"):
        run_demo_scenario(choice)
    elif choice == "5":
        interactive_chat()
    elif choice == "6":
        for key in ("1", "2", "3", "4"):
            run_demo_scenario(key)
    else:
        print("Invalid choice. Running all demos.")
        for key in ("1", "2", "3", "4"):
            run_demo_scenario(key)

In [ ]:
main()